In [1]:
import os
import pandas as pd
import subprocess
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from openpyxl import Workbook
from openpyxl.styles import Alignment
from openpyxl.utils.dataframe import dataframe_to_rows


# =========================
# Chrome配置（和B站一致）
# =========================

options = Options()


# 保存登录状态
options.add_argument(
    r"--user-data-dir=C:\Users\12082\AppData\Local\Google\Chrome\SeleniumData"
)


options.add_argument("--start-maximized")


# 创建Chrome

try:

    driver = webdriver.Chrome(
        service=Service(
            ChromeDriverManager().install()
        ),
        options=options
    )

    print("ChromeDriver加载成功！")


except Exception as e:

    print(
        f"ChromeDriver初始化失败:{e}"
    )

    exit()



# =========================
# 打开微信视频号后台
# =========================

url = "https://channels.weixin.qq.com/platform/post/list"


try:

    driver.get(url)

    print(
        "成功进入目标页面！"
    )


except Exception as e:

    print(
        f"打开页面失败:{e}"
    )

    driver.quit()

    exit()



# =========================
# 等待登录
# =========================

wait = WebDriverWait(driver, 600)


try:

    print(
        "请扫描二维码登录..."
    )


    wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                "div.post-feed-item"
            )
        )
    )


    print(
        "扫码成功，开始提取数据！"
    )


except Exception as e:

    print(
        f"登录失败或超时:{e}"
    )


    driver.quit()

    exit()



# =========================
# 数字转换函数
# =========================

def convert_to_number(value):

    if not value:
        return 0


    value = value.strip()


    if "万" in value:

        try:

            return int(
                float(
                    value.replace("万","")
                )
                * 10000
            )

        except:

            return 0


    try:

        return int(
            value.replace(",","")
        )

    except:

        return 0



# =========================
# 提取数据
# =========================

def extract_video_data():


    video_data = []


    try:


        WebDriverWait(
            driver,
            20
        ).until(

            EC.presence_of_all_elements_located(
                (
                    By.CSS_SELECTOR,
                    "div.post-feed-item"
                )
            )

        )


        script = """

        let data = [];


        document.querySelectorAll(
            'div.post-feed-item'
        ).forEach(item => {


            let title =
            item.querySelector(
                'div.post-title'
            )?.innerText || '';


            let date =
            item.querySelector(
                'div.post-time'
            )?.innerText.split(' ')[0] || '';


            let views =
            item.querySelector(
                'div.data-item:nth-child(1) span.count'
            )?.innerText || '';


            let likes =
            item.querySelector(
                'div.data-item:nth-child(2) span.count'
            )?.innerText || '';


            let comments =
            item.querySelector(
                'div.data-item:nth-child(3) span.count'
            )?.innerText || '';


            let shares =
            item.querySelector(
                'div.data-item:nth-child(4) span.count'
            )?.innerText || '';


            let votes =
            item.querySelector(
                'div.data-item:nth-child(5) span.count'
            )?.innerText || '';


            data.push(
            {
                title,
                date,
                views,
                likes,
                comments,
                shares,
                votes
            });


        });


        return data;

        """


        video_data = driver.execute_script(
            script
        )


        print(
            "JS返回数据:"
        )

        print(
            video_data[:3]
        )


        if video_data:

            print(
                "成功提取视频数据！"
            )

        else:

            print(
                "未提取到数据"
            )


    except Exception as e:


        print(
            f"提取错误:{e}"
        )


    return video_data



# =========================
# 保存Excel
# =========================

def save_to_excel(video_data):


    if not video_data:

        print(
            "没有数据保存"
        )

        return



    df = pd.DataFrame(
        video_data
    )


    df.rename(
        columns={

            "title":"标题",
            "date":"日期",
            "views":"播放量",
            "likes":"喜欢",
            "comments":"评论",
            "shares":"转发",
            "votes":"点赞"

        },
        inplace=True
    )


    for col in [
        "播放量",
        "喜欢",
        "评论",
        "转发",
        "点赞"
    ]:

        df[col] = df[col].apply(
            convert_to_number
        )



    file_path = os.path.join(
        os.path.expanduser("~"),
        "Desktop",
        "微信视频信息.xlsx"
    )



    wb = Workbook()

    ws = wb.active

    ws.title = "视频信息"



    for row in dataframe_to_rows(
        df,
        index=False,
        header=True
    ):

        ws.append(row)



    ws.column_dimensions["A"].width = 40

    ws.column_dimensions["B"].width = 20


    for col in [
        "C",
        "D",
        "E",
        "F",
        "G"
    ]:

        ws.column_dimensions[col].width = 10



    for row in ws.iter_rows(
        min_row=2,
        max_row=ws.max_row,
        min_col=2,
        max_col=7
    ):

        for cell in row:

            cell.alignment = Alignment(
                horizontal="right"
            )


    wb.save(file_path)


    print(
        f"保存成功:{file_path}"
    )


    os.startfile(
        file_path
    )



# =========================
# 主程序
# =========================


if __name__ == "__main__":


    data = extract_video_data()


    save_to_excel(data)


    driver.quit()

ChromeDriver加载成功！
成功进入目标页面！
请扫描二维码登录...
登录失败或超时:Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=150.0.7871.114)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7c5cca865+152a5]
	chromedriver!GetHandleVerifier [0x7ff7c5cca8c0+15300]
	chromedriver!(No symbol) [0x7ff7c582596d]
	chromedriver!(No symbol) [0x7ff7c5811132]
	chromedriver!(No symbol) [0x7ff7c583706c]
	chromedriver!(No symbol) [0x7ff7c58b0430]
	chromedriver!(No symbol) [0x7ff7c58cd5b2]
	chromedriver!(No symbol) [0x7ff7c5872b3c]
	chromedriver!(No symbol) [0x7ff7c5873a53]
	chromedriver!GetHandleVerifier [0x7ff7c62ad3a1+5f7de1]
	chromedriver!GetHandleVerifier [0x7ff7c62a7a3b+5f247b]
	chromedriver!GetHandleVerifier [0x7ff7c62cbca5+6166e5]
	chromedriver!GetHandleVerifier [0x7ff7c5ce72ae+31cee]
	chromedriver!GetHandleVerifier [0x7ff7c5cefe3c+3a87c]
	chromedriver!GetHandleVerifier [0x7ff7c5cd4404+1ee44]
	chromedriver!GetHandleVe